# 💳 Kredi Kartı Dolandırıcılık Tespiti
## Makine Öğrenmesi Projesi — Uçtan Uca Pipeline

**Veri Seti:** [Kaggle — Credit Card Fraud Detection (ULB)](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)  
**Problem Türü:** Binary Classification  
**Öncelikli Metrikler:** PR-AUC, F1 Score, Recall  

---

### Proje Aşamaları
1. Problem Tanımı
2. Veri Seti Tanıtımı
3. EDA (Keşifsel Veri Analizi)
4. Veri Kalitesi
5. Preprocessing & Feature Engineering
6. Veri Bölme & Deney Tasarımı
7. Baseline Model
8. İlk Model Karşılaştırması
9. Aday Model Seçimi
10. Hiperparametre Optimizasyonu
11. Güçlü Rakip Karşılaştırması
12. Final Model Değerlendirme

---
## ⚙️ Kütüphaneler

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, average_precision_score,
    f1_score, precision_score, recall_score,
    ConfusionMatrixDisplay, PrecisionRecallDisplay, RocCurveDisplay
)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

try:
    import xgboost as xgb
    import lightgbm as lgb
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
except ImportError:
    print('XGBoost / LightGBM / Optuna kurulu değil. Kurmak için:')
    print('pip install xgboost lightgbm optuna imbalanced-learn')

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
print('✅ Kütüphaneler yüklendi.')

---
## 📌 Aşama 1 — Problem Tanımı

Kredi kartı dolandırıcılığı, finans sektöründe her yıl milyarlarca dolarlık kayba yol açmaktadır. Bu projede, bir Avrupa bankasına ait gerçek kredi kartı işlem verisini kullanarak dolandırıcılık olan ve olmayan işlemleri otomatik olarak sınıflandıran bir makine öğrenmesi sistemi geliştirilecektir.

**Hedef kullanıcı:** Bankalar ve finans kurumlarının risk/uyum ekipleri  
**Başarı kriteri:** PR-AUC ve Macro-F1 skorunun maksimize edilmesi  
**Not:** Veri setindeki ciddi sınıf dengesizliği (%0.17 dolandırıcılık) nedeniyle Accuracy tek başına yeterli metrik kabul edilmeyecektir.

---
## 📂 Aşama 2 — Veri Seti Tanıtımı

In [ ]:
df = pd.read_csv('creditcard.csv')

print('=' * 50)
print('VERİ SETİ GENEL BİLGİLER')
print('=' * 50)
print(f'Kaynak  : Kaggle — ULB Machine Learning Group')
print(f'Lisans  : DbCL v1.0')
print(f'Boyut   : {df.shape[0]:,} satır x {df.shape[1]} sütun')
print(f'Bellek  : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')
print()
print('Sütunlar:')
print('  - Time   : İlk işlemden bu yana geçen saniye')
print('  - V1–V28 : PCA ile anonimleştirilmiş özellikler')
print('  - Amount : İşlem tutarı (€)')
print('  - Class  : 0 = Normal, 1 = Dolandırıcılık (hedef)')
print()
print('Sınıf dağılımı:')
print(df['Class'].value_counts())
print(f'\nDolandırıcılık oranı: %{df["Class"].mean()*100:.4f}')
display(df.head())

---
## 🔍 Aşama 3 — EDA (Keşifsel Veri Analizi)

In [ ]:
# 3.1 Genel istatistikler
display(df.describe().round(2))

In [ ]:
# 3.2 Class dağılımı
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['Class'].value_counts()
axes[0].bar(['Normal (0)', 'Dolandırıcılık (1)'],
            counts.values, color=['steelblue', 'crimson'])
axes[0].set_title('İşlem Sayısı')
axes[0].set_ylabel('Adet')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 1000, f'{v:,}', ha='center', fontsize=10)

axes[1].pie(counts.values,
            labels=[f'Normal\n({counts[0]:,})', f'Dolandırıcılık\n({counts[1]:,})'],
            autopct='%1.3f%%',
            colors=['steelblue', 'crimson'],
            startangle=90)
axes[1].set_title('İşlem Oranı')

plt.suptitle('Class Dağılımı — Ciddi Dengesizlik (%0.17 dolandırıcılık)', fontsize=13)
plt.tight_layout()
plt.show()
print(f'Normal: {counts[0]:,} | Dolandırıcılık: {counts[1]:,}')
print(f'Oran: 1 dolandırıcılığa karşı {counts[0]//counts[1]} normal işlem')

In [ ]:
# 3.3 Amount dağılımı
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, color, name in [(0, 'steelblue', 'Normal'), (1, 'crimson', 'Dolandırıcılık')]:
    axes[0].hist(df[df['Class'] == label]['Amount'].clip(upper=500),
                 bins=50, alpha=0.6, color=color, label=name)
axes[0].set_title('Amount Dağılımı (0–500€)')
axes[0].set_xlabel('İşlem Tutarı (€)')
axes[0].set_ylabel('Frekans')
axes[0].legend()

df[df['Amount'] < 500].boxplot(column='Amount', by='Class', ax=axes[1],
                                patch_artist=True,
                                boxprops=dict(facecolor='lightblue'))
axes[1].set_title('Amount — Normal vs Dolandırıcılık')
axes[1].set_xlabel('Class (0=Normal, 1=Dolandırıcılık)')
axes[1].set_ylabel('İşlem Tutarı (€)')
plt.suptitle('')
plt.tight_layout()
plt.show()

print('Amount özeti — Dolandırıcılık:')
print(df[df['Class']==1]['Amount'].describe().round(2))

In [ ]:
# 3.4 Time dağılımı
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for label, color, name in [(0, 'steelblue', 'Normal'), (1, 'crimson', 'Dolandırıcılık')]:
    axes[0].hist(df[df['Class']==label]['Time'] / 3600,
                 bins=48, alpha=0.6, color=color, label=name, density=True)
axes[0].set_title('İşlem Zamanı Dağılımı (saat)')
axes[0].set_xlabel('Saat (0–48, 2 günlük veri)')
axes[0].set_ylabel('Yoğunluk')
axes[0].legend()
axes[0].axvline(24, color='gray', linestyle='--', alpha=0.5)

df['Hour'] = (df['Time'] // 3600) % 24
hourly = df.groupby('Hour')['Class'].mean() * 100
axes[1].bar(hourly.index, hourly.values, color='crimson', alpha=0.7)
axes[1].set_title('Saate Göre Dolandırıcılık Oranı (%)')
axes[1].set_xlabel('Saat')
axes[1].set_ylabel('Dolandırıcılık oranı (%)')

plt.tight_layout()
plt.show()

In [ ]:
# 3.5 En önemli özellikler — boxplot
top_features = ['V17', 'V14', 'V12', 'V10', 'V16', 'V3']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, feat in zip(axes.flat, top_features):
    sns.boxplot(data=df, x='Class', y=feat,
                hue='Class', ax=ax,
                palette={0: 'steelblue', 1: 'crimson'},
                legend=False)
    ax.set_title(f'{feat} vs Class')
    ax.set_xlabel('Class (0=Normal, 1=Dolandırıcılık)')

plt.suptitle('En Önemli Özellikler — Normal vs Dolandırıcılık', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# 3.6 Korelasyon ısı haritası
important_cols = ['V17','V14','V12','V10','V16','V3','V11','V4','Amount','Class']

plt.figure(figsize=(10, 8))
corr_matrix = df[important_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            linewidths=0.5)
plt.title('Önemli Özellikler — Korelasyon Matrisi')
plt.tight_layout()
plt.show()

### 📝 EDA Bulguları

- **Class dengesizliği:** %0.17 dolandırıcılık — Accuracy yanıltıcı, PR-AUC ve F1 öncelikli metrik olmalı
- **Amount:** Dolandırıcılık işlemleri genellikle düşük-orta tutarlı; sağa çarpık dağılım → log/robust scaling gerekli
- **Time:** Gece saatlerinde (00:00–06:00) dolandırıcılık oranı görece yüksek
- **V17, V14, V12:** Class ile en güçlü korelasyon — model için önemli sinyaller

---
## 🧹 Aşama 4 — Veri Kalitesi

In [ ]:
print('=' * 50)
print('VERİ KALİTESİ RAPORU')
print('=' * 50)

# Eksik değer
missing = df.isnull().sum()
print(f'\n1. Eksik değer: {missing.sum()} (✅ Yok)')

# Duplicate
dupes = df.duplicated().sum()
print(f'2. Tekrarlı satır: {dupes}')
if dupes > 0:
    print('   → Duplicate satırlar kaldırılacak')
    df = df.drop_duplicates()
    print(f'   → Yeni boyut: {df.shape}')

# Outlier (IQR yöntemi — Amount)
Q1 = df['Amount'].quantile(0.25)
Q3 = df['Amount'].quantile(0.75)
IQR = Q3 - Q1
outliers = ((df['Amount'] < Q1 - 1.5*IQR) | (df['Amount'] > Q3 + 1.5*IQR)).sum()
print(f'3. Amount outlier (IQR): {outliers:,} ({outliers/len(df)*100:.2f}%)')
print('   → RobustScaler ile ele alınacak (outlier silmek yerine)')

# Sınıf dengesizliği
print(f'\n4. Sınıf dengesizliği:')
print(f'   Normal     : {(df["Class"]==0).sum():,}')
print(f'   Dolandırıcı: {(df["Class"]==1).sum():,}')
print(f'   → SMOTE ile oversampling yapılacak (sadece train setinde!)')

print('\n✅ Veri kalitesi kontrolü tamamlandı.')

---
## ⚙️ Aşama 5 — Preprocessing & Feature Engineering

In [ ]:
# Feature Engineering
df['Amount_log'] = np.log1p(df['Amount'])          # Log dönüşümü
df['Time_hour']  = (df['Time'] // 3600) % 24        # Saatlik özellik
df['Time_day']   = (df['Time'] // 86400).astype(int) # Gün bilgisi

# Özellik ve hedef
drop_cols = ['Time', 'Amount', 'Hour']  # Orijinal Time ve Amount'u çıkar
feature_cols = [c for c in df.columns if c not in drop_cols + ['Class']]

X = df[feature_cols]
y = df['Class']

print(f'Özellik sayısı: {X.shape[1]}')
print(f'Özellikler: {feature_cols}')
print(f'Hedef dağılımı: \n{y.value_counts()}')

---
## ✂️ Aşama 6 — Veri Bölme & Deney Tasarımı

> ⚠️ **Data Leakage Kontrolü:** SMOTE ve Scaler yalnızca train setinde uygulanacak.  
> Test seti yalnızca final değerlendirme için kullanılacak, hiçbir şekilde model seçimi veya tuning'de kullanılmayacak.

In [ ]:
# Önce %20 test setini ayır ve kilitle
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

# Kalan %80'i train/val olarak böl (%64 train, %16 val)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.20, random_state=RANDOM_STATE, stratify=y_temp
)

print('Veri bölme tamamlandı:')
print(f'  Train : {X_train.shape[0]:,} ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'  Val   : {X_val.shape[0]:,} ({X_val.shape[0]/len(X)*100:.1f}%)')
print(f'  Test  : {X_test.shape[0]:,} ({X_test.shape[0]/len(X)*100:.1f}%)')
print()
print(f'Train dolandırıcılık oranı: %{y_train.mean()*100:.3f}')
print(f'Val   dolandırıcılık oranı: %{y_val.mean()*100:.3f}')
print(f'Test  dolandırıcılık oranı: %{y_test.mean()*100:.3f}')

# CV stratejisi
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
print(f'\nCV: StratifiedKFold(n_splits=5)')

---
## 🏁 Aşama 7 — Baseline Model

In [ ]:
# Scaler — sadece train'de fit
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

# Dummy Baseline — her şeyi çoğunluk sınıfı (0) olarak tahmin et
dummy = DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE)
dummy.fit(X_train_scaled, y_train)
y_pred_dummy = dummy.predict(X_val_scaled)

print('BASELINE — Majority Class (her şeyi Normal tahmin et)')
print('=' * 50)
print(f'Accuracy : {(y_pred_dummy == y_val).mean():.4f}')
print(f'F1 Score : {f1_score(y_val, y_pred_dummy, average="macro"):.4f}')
print(f'Recall   : {recall_score(y_val, y_pred_dummy, zero_division=0):.4f}')
print()
print('⚠️  Accuracy yüksek görünüyor ama F1=0 — yanıltıcı metrik!')
print(classification_report(y_val, y_pred_dummy, zero_division=0))

---
## ⚖️ Aşama 8 — İlk Model Karşılaştırması

In [ ]:
# SMOTE — sadece train setinde uygula
smote = SMOTE(random_state=RANDOM_STATE)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)
print(f'SMOTE sonrası train: {X_train_res.shape[0]:,} ({y_train_res.value_counts().to_dict()})')

# Modeller
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'Decision Tree':       DecisionTreeClassifier(random_state=RANDOM_STATE),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1),
    'Naive Bayes':         GaussianNB(),
    'KNN':                 KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
}

results = []
for name, model in models.items():
    model.fit(X_train_res, y_train_res)
    y_pred = model.predict(X_val_scaled)
    y_prob = model.predict_proba(X_val_scaled)[:, 1] if hasattr(model, 'predict_proba') else None

    res = {
        'Model'    : name,
        'Accuracy' : round((y_pred == y_val).mean(), 4),
        'F1'       : round(f1_score(y_val, y_pred, average='macro'), 4),
        'Recall'   : round(recall_score(y_val, y_pred), 4),
        'Precision': round(precision_score(y_val, y_pred, zero_division=0), 4),
        'PR-AUC'   : round(average_precision_score(y_val, y_prob), 4) if y_prob is not None else None,
    }
    results.append(res)
    print(f'✓ {name}')

results_df = pd.DataFrame(results).sort_values('PR-AUC', ascending=False)
display(results_df)

In [ ]:
# Görsel karşılaştırma
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

metrics = ['F1', 'PR-AUC']
colors = ['steelblue', 'crimson']

for ax, metric, color in zip(axes, metrics, colors):
    sorted_df = results_df.dropna(subset=[metric]).sort_values(metric)
    ax.barh(sorted_df['Model'], sorted_df[metric], color=color, alpha=0.8)
    ax.set_title(f'Model Karşılaştırması — {metric}')
    ax.set_xlabel(metric)
    for i, (_, row) in enumerate(sorted_df.iterrows()):
        ax.text(row[metric] + 0.005, i, f'{row[metric]:.4f}', va='center')

plt.tight_layout()
plt.show()

---
## 🎯 Aşama 9 — Aday Model Seçimi

Karşılaştırma tablosuna göre en yüksek PR-AUC ve F1 skoruna sahip model seçilir.  
Genellikle **Random Forest** veya **LightGBM/XGBoost** öne çıkar.

In [ ]:
best_model_name = results_df.iloc[0]['Model']
print(f'Seçilen aday model: {best_model_name}')
print(f'PR-AUC: {results_df.iloc[0]["PR-AUC"]}')
print(f'F1    : {results_df.iloc[0]["F1"]}')
print()
print('Gerekçe: PR-AUC ve F1 metrikleri dengesiz veri setinde daha anlamlı.')
print('Sadece yüksek metrik değil, problem bağlamı (dolandırıcılık tespiti) gözetildi.')

---
## 🔧 Aşama 10 — Hiperparametre Optimizasyonu (Optuna)

In [ ]:
import optuna
from sklearn.ensemble import RandomForestClassifier

def objective(trial):
    params = {
        'n_estimators'      : trial.suggest_int('n_estimators', 50, 300),
        'max_depth'         : trial.suggest_int('max_depth', 3, 20),
        'min_samples_split' : trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf'  : trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features'      : trial.suggest_categorical('max_features', ['sqrt', 'log2']),
        'random_state'      : RANDOM_STATE,
        'n_jobs'            : -1,
    }
    model = RandomForestClassifier(**params)
    model.fit(X_train_res, y_train_res)
    y_prob = model.predict_proba(X_val_scaled)[:, 1]
    return average_precision_score(y_val, y_prob)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30, show_progress_bar=True)

print(f'\nEn iyi PR-AUC: {study.best_value:.4f}')
print(f'En iyi parametreler: {study.best_params}')

In [ ]:
# Optuna sonuçlarını görselleştir
optuna.visualization.matplotlib.plot_optimization_history(study)
plt.title('Optuna — Optimizasyon Tarihi')
plt.tight_layout()
plt.show()

In [ ]:
# Optimize edilmiş modeli eğit
best_params = study.best_params
best_params['random_state'] = RANDOM_STATE
best_params['n_jobs'] = -1

optimized_model = RandomForestClassifier(**best_params)
optimized_model.fit(X_train_res, y_train_res)

y_pred_opt = optimized_model.predict(X_val_scaled)
y_prob_opt = optimized_model.predict_proba(X_val_scaled)[:, 1]

print('Optimize edilmiş model — Validasyon sonuçları:')
print(f'PR-AUC   : {average_precision_score(y_val, y_prob_opt):.4f}')
print(f'F1 Macro : {f1_score(y_val, y_pred_opt, average="macro"):.4f}')
print(f'Recall   : {recall_score(y_val, y_pred_opt):.4f}')
print(f'Precision: {precision_score(y_val, y_pred_opt, zero_division=0):.4f}')

---
## 🏆 Aşama 11 — Güçlü Rakip Karşılaştırması

5 güçlü rakiple karşılaştırma yapılır.

In [ ]:
import xgboost as xgb
import lightgbm as lgb

strong_models = {
    'Random Forest (Tuned)' : optimized_model,
    'XGBoost'               : xgb.XGBClassifier(
                                  n_estimators=200, max_depth=6,
                                  learning_rate=0.05, subsample=0.8,
                                  use_label_encoder=False,
                                  eval_metric='aucpr',
                                  random_state=RANDOM_STATE, n_jobs=-1),
    'LightGBM'              : lgb.LGBMClassifier(
                                  n_estimators=200, max_depth=8,
                                  learning_rate=0.05, num_leaves=31,
                                  random_state=RANDOM_STATE, n_jobs=-1,
                                  verbose=-1),
    'Gradient Boosting'     : GradientBoostingClassifier(
                                  n_estimators=100, max_depth=5,
                                  learning_rate=0.1, random_state=RANDOM_STATE),
    'Logistic Regression'   : LogisticRegression(
                                  C=0.01, max_iter=1000,
                                  random_state=RANDOM_STATE),
    'SVM (RBF)'             : SVC(kernel='rbf', probability=True,
                                  C=1.0, gamma='scale',
                                  random_state=RANDOM_STATE),
}

strong_results = []
for name, model in strong_models.items():
    model.fit(X_train_res, y_train_res)
    y_pred = model.predict(X_val_scaled)
    y_prob = model.predict_proba(X_val_scaled)[:, 1]

    strong_results.append({
        'Model'    : name,
        'Accuracy' : round((y_pred == y_val).mean(), 4),
        'F1'       : round(f1_score(y_val, y_pred, average='macro'), 4),
        'Recall'   : round(recall_score(y_val, y_pred), 4),
        'Precision': round(precision_score(y_val, y_pred, zero_division=0), 4),
        'PR-AUC'   : round(average_precision_score(y_val, y_prob), 4),
        'ROC-AUC'  : round(roc_auc_score(y_val, y_prob), 4),
    })
    print(f'✓ {name}')

strong_df = pd.DataFrame(strong_results).sort_values('PR-AUC', ascending=False)
display(strong_df)

In [ ]:
# Güçlü rakip karşılaştırma grafiği
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, metric in zip(axes, ['PR-AUC', 'F1', 'Recall']):
    sorted_df = strong_df.sort_values(metric)
    colors_bar = ['gold' if m == 'Random Forest (Tuned)' else 'steelblue'
                  for m in sorted_df['Model']]
    ax.barh(sorted_df['Model'], sorted_df[metric], color=colors_bar, alpha=0.85)
    ax.set_title(f'{metric}')
    for i, (_, row) in enumerate(sorted_df.iterrows()):
        ax.text(row[metric] + 0.002, i, f'{row[metric]:.4f}', va='center', fontsize=9)

plt.suptitle('Güçlü Rakip Karşılaştırması (Altın = Bizim Modelimiz)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# PR Curve — tüm modeller
plt.figure(figsize=(10, 6))
for name, model in strong_models.items():
    y_prob = model.predict_proba(X_val_scaled)[:, 1]
    PrecisionRecallDisplay.from_predictions(
        y_val, y_prob, name=name, ax=plt.gca()
    )
plt.title('Precision-Recall Curve — Tüm Modeller')
plt.legend(loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()

---
## 🏅 Aşama 12 — Final Model Değerlendirme

> ⚠️ Test seti ilk kez burada kullanılıyor!

In [ ]:
# Final model: Rakip karşılaştırmasından en iyi çıkan model
final_model_name = strong_df.iloc[0]['Model']
final_model = strong_models[final_model_name]

print(f'Final model: {final_model_name}')
print('=' * 55)

# TEST SETİNDE DEĞERLENDİR
y_pred_final = final_model.predict(X_test_scaled)
y_prob_final = final_model.predict_proba(X_test_scaled)[:, 1]

print(f'PR-AUC   : {average_precision_score(y_test, y_prob_final):.4f}')
print(f'ROC-AUC  : {roc_auc_score(y_test, y_prob_final):.4f}')
print(f'F1 Macro : {f1_score(y_test, y_pred_final, average="macro"):.4f}')
print(f'Recall   : {recall_score(y_test, y_pred_final):.4f}')
print(f'Precision: {precision_score(y_test, y_pred_final, zero_division=0):.4f}')
print(f'Accuracy : {(y_pred_final == y_test).mean():.4f}')
print()
print(classification_report(y_test, y_pred_final))

In [ ]:
# Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_final, ax=axes[0],
    display_labels=['Normal', 'Dolandırıcılık'],
    cmap='Blues'
)
axes[0].set_title(f'Confusion Matrix — {final_model_name}')

RocCurveDisplay.from_predictions(y_test, y_prob_final, ax=axes[1])
axes[1].set_title('ROC Curve — Test Seti')

plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance
if hasattr(final_model, 'feature_importances_'):
    importance_df = pd.DataFrame({
        'Feature'   : feature_cols,
        'Importance': final_model.feature_importances_
    }).sort_values('Importance', ascending=False).head(15)

    plt.figure(figsize=(10, 6))
    sns.barplot(data=importance_df, x='Importance', y='Feature', palette='viridis')
    plt.title('Feature Importance — Top 15')
    plt.tight_layout()
    plt.show()

---
## 💾 Model Kaydetme

In [ ]:
import joblib

joblib.dump(final_model, 'model.pkl')
joblib.dump(scaler,      'scaler.pkl')

print('✅ Model ve scaler kaydedildi: model.pkl, scaler.pkl')

---
## 📊 Aşama 12 Özet — Güçlü ve Zayıf Yönler

### Güçlü Yönler
- Yüksek PR-AUC skoru dengesiz veri setinde gerçek anlamda başarı gösteriyor
- SMOTE + RobustScaler pipeline'ı data leakage'ı önlüyor
- Optuna ile sistematik hiperparametre optimizasyonu yapıldı
- 5+ güçlü rakiple adil karşılaştırma yapıldı

### Zayıf Yönler / Sınırlılıklar
- V1–V28 PCA sonrası — özellik yorumlanabilirliği kısıtlı
- Sadece 2 günlük Avrupa verisi — farklı coğrafyaya genelleme güç
- Gerçek zamanlı tahmin için model latency optimize edilmedi

### Gelecek Çalışmalar
- SHAP değerleri ile model açıklanabilirliği
- Zaman bazlı cross-validation (temporal split)
- Deep learning yaklaşımları (Autoencoder tabanlı anomaly detection)